In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import pickle
import os

df = pd.read_csv(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\data\clean_ratings.csv')
print(df.shape)
print(df.head(3))

(72696, 6)
          user_id  product_id  rating  \
0  A3FO5AKVTFRCRJ  0739079891     5.0   
1  A1H4I7VO8YVMEI  0739079891     1.0   
2  A1NZF2Y9MQD5QA  0739079891     4.0   

                                         review_text  \
0                            It's good for beginners   
1  Great concept with great instructions and CDs....   
2                                good starter set up   

                     summary   timestamp  
0                 Five Stars  1477785600  
1  Frustratingly out of tune  1422403200  
2                 Four Stars  1420329600  


In [2]:
# For content-based filtering we need ONE row per product
# We'll aggregate all reviews for each product into a single text profile

product_profiles = df.groupby('product_id').agg(
    all_reviews=('review_text', lambda x: ' '.join(x.fillna(''))),
    all_summaries=('summary', lambda x: ' '.join(x.fillna(''))),
    avg_rating=('rating', 'mean'),
    review_count=('user_id', 'count')
).reset_index()

# Combine reviews + summaries into one rich text field
# Summaries are more signal-dense so we repeat them 3x (boosting)
product_profiles['content'] = (
    product_profiles['all_summaries'] * 3 + ' ' +
    product_profiles['all_reviews']
)

print(f"Product profiles: {product_profiles.shape}")
print(product_profiles[['product_id', 'avg_rating', 'review_count']].head())

Product profiles: (2717, 6)
   product_id  avg_rating  review_count
0  0739079891    3.866667            15
1  1480360295    3.500000            20
2  9792372326    4.444444             9
3  B00000J50W    4.666667             6
4  B00001W0DH    4.600000             5


In [3]:
# TF-IDF converts text into numbers
# TF = how often a word appears in THIS product's reviews
# IDF = penalizes words that appear in ALL products (not useful)
# Result: each product becomes a vector of important words

tfidf = TfidfVectorizer(
    max_features=5000,    # top 5000 most important words
    stop_words='english', # remove "the", "is", "and" etc
    ngram_range=(1, 2),   # include single words AND pairs ("battery life")
    min_df=2,             # ignore words appearing in less than 2 products
    sublinear_tf=True     # dampen effect of very frequent words
)

tfidf_matrix = tfidf.fit_transform(product_profiles['content'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
# Should be (n_products, 5000)

TF-IDF matrix shape: (2717, 5000)


In [4]:
# Pure text similarity ignores product quality
# A product with similar reviews but 1-star avg rating shouldn't rank high
# We blend TF-IDF similarity with normalized avg rating

from scipy.sparse import hstack, csr_matrix as csr

# Normalize avg_rating to 0-1 scale
scaler = MinMaxScaler()
rating_scores = scaler.fit_transform(
    product_profiles[['avg_rating', 'review_count']]
)

# Convert to sparse and append to tfidf matrix
rating_sparse = csr(rating_scores)
combined_matrix = hstack([tfidf_matrix * 0.8, rating_sparse * 0.2])

print("Combined matrix shape:", combined_matrix.shape)

Combined matrix shape: (2717, 5002)


In [5]:
# Build a product index for fast lookup
product_idx_map = {pid: idx for idx, pid in enumerate(product_profiles['product_id'])}
product_id_map = {idx: pid for pid, idx in product_idx_map.items()}

def get_similar_products(product_id, n=10):
    if product_id not in product_idx_map:
        print(f"Product {product_id} not found")
        return []

    idx = product_idx_map[product_id]

    # Compute cosine similarity between this product and ALL others
    product_vector = combined_matrix[idx]
    similarity_scores = cosine_similarity(product_vector, combined_matrix).flatten()

    # Sort by similarity, exclude itself
    similar_indices = similarity_scores.argsort()[::-1][1:n+1]

    results = []
    for sim_idx in similar_indices:
        pid = product_id_map[sim_idx]
        results.append({
            'product_id': pid,
            'similarity_score': round(float(similarity_scores[sim_idx]), 4),
            'avg_rating': round(product_profiles.iloc[sim_idx]['avg_rating'], 2),
            'review_count': int(product_profiles.iloc[sim_idx]['review_count'])
        })

    return results

# Test it
sample_product = df['product_id'].iloc[0]
print(f"Content-based similar products to: {sample_product}\n")
results = get_similar_products(sample_product)
for i, r in enumerate(results, 1):
    print(f"{i}. {r['product_id']} | similarity: {r['similarity_score']} | rating: {r['avg_rating']} | reviews: {r['review_count']}")

Content-based similar products to: 0739079891

1. B00169KU7C | similarity: 0.4435 | rating: 4.28 | reviews: 92
2. B002ZSE9ES | similarity: 0.4315 | rating: 4.65 | reviews: 20
3. B00172UV6S | similarity: 0.4241 | rating: 3.88 | reviews: 16
4. B001731R6A | similarity: 0.4191 | rating: 4.31 | reviews: 32
5. B0018TF0O8 | similarity: 0.3954 | rating: 3.71 | reviews: 7
6. B000RVYMWE | similarity: 0.3884 | rating: 4.08 | reviews: 13
7. B001EL6I8W | similarity: 0.3725 | rating: 4.11 | reviews: 19
8. B003AQ4UDY | similarity: 0.366 | rating: 4.6 | reviews: 5
9. B002RGPQJ0 | similarity: 0.341 | rating: 4.3 | reviews: 33
10. B000S8CX7M | similarity: 0.3317 | rating: 2.79 | reviews: 19


In [6]:
# For a given user, find products they rated highly
# Then find products similar to those — that's content-based recommendation

def recommend_for_user_content(user_id, n=10):
    # Get products this user rated 4 or 5 stars
    user_ratings = df[(df['user_id'] == user_id) & (df['rating'] >= 4)]

    if user_ratings.empty:
        print(f"No high ratings found for user {user_id}")
        return []

    # Aggregate similarity scores across all liked products
    score_accumulator = np.zeros(len(product_profiles))

    for _, row in user_ratings.iterrows():
        pid = row['product_id']
        if pid not in product_idx_map:
            continue
        idx = product_idx_map[pid]
        product_vector = combined_matrix[idx]
        sims = cosine_similarity(product_vector, combined_matrix).flatten()
        score_accumulator += sims * row['rating']  # weight by rating

    # Zero out products user already rated
    rated_products = set(df[df['user_id'] == user_id]['product_id'])
    for pid in rated_products:
        if pid in product_idx_map:
            score_accumulator[product_idx_map[pid]] = 0

    # Get top N
    top_indices = score_accumulator.argsort()[::-1][:n]

    results = []
    for idx in top_indices:
        results.append({
            'product_id': product_id_map[idx],
            'score': round(float(score_accumulator[idx]), 4),
            'avg_rating': round(float(product_profiles.iloc[idx]['avg_rating']), 2)
        })

    return results

# Test it
sample_user = df['user_id'].iloc[0]
print(f"Content-based recommendations for: {sample_user}\n")
recs = recommend_for_user_content(sample_user)
for i, r in enumerate(recs, 1):
    print(f"{i}. {r['product_id']} | score: {r['score']} | avg_rating: {r['avg_rating']}")

Content-based recommendations for: A3FO5AKVTFRCRJ

1. B0002H0A3S | score: 16.8081 | avg_rating: 4.71
2. B0007Y09VO | score: 16.8064 | avg_rating: 4.71
3. B0002E1NNC | score: 16.1643 | avg_rating: 4.7
4. B0002E1NWI | score: 16.1643 | avg_rating: 4.7
5. B0002E1J3Q | score: 13.9683 | avg_rating: 4.76
6. B0002H04NE | score: 13.4703 | avg_rating: 4.65
7. B0006IQLF4 | score: 13.4703 | avg_rating: 4.65
8. B0002H0JZ2 | score: 12.8808 | avg_rating: 4.62
9. B0002DV6TO | score: 12.8604 | avg_rating: 4.48
10. B0002DUS8E | score: 12.8575 | avg_rating: 4.68


In [7]:
os.makedirs('../models', exist_ok=True)

with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('../models/combined_matrix.pkl', 'wb') as f:
    pickle.dump(combined_matrix, f)

with open('../models/product_profiles.pkl', 'wb') as f:
    pickle.dump(product_profiles, f)

with open('../models/product_idx_map.pkl', 'wb') as f:
    pickle.dump({'product_idx_map': product_idx_map, 'product_id_map': product_id_map}, f)

print("✅ Content-based model saved to models/")
print("\nModels saved so far:")
for f in os.listdir('../models'):
    print(f" - {f}")

✅ Content-based model saved to models/

Models saved so far:
 - als_model.pkl
 - combined_matrix.pkl
 - encoders.pkl
 - interaction_matrix.pkl
 - product_idx_map.pkl
 - product_profiles.pkl
 - tfidf_vectorizer.pkl
